In [5]:
from committor_control import *

import pandas as pd
import numpy as np
import os, sys, re
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path

plt.rcParams['xtick.labelsize'] = 14  # font size
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams["font.family"] = "serif"
mpl.rcParams['pdf.fonttype'] = 42  # Use TrueType fonts (editable in Illustrator)
mpl.rcParams['ps.fonttype'] = 42   # Same for EPS
mpl.rcParams['svg.fonttype'] = 'none'  # Keep text as text in SVG
mpl.rcParams["mathtext.fontset"] =  "cm"

In [ ]:
if __name__ == "__main__":
    dt = 0.01
    num_trajs = 1000
 
    sys_dw   = make_double_well_curved(lambda_u=2.0, lambda_s=8.0, c=0.5)
    sys_lin  = make_linear_system(lambda_u=2.0, lambda_s=-8.0)
    sys_pend = make_pendulum(g=1.5, length=1.0, damping=0.0)
 
    use_whitened = True
 
    systems = {
        "linear_2d":   (sys_lin,  1.0, np.inf, 10, 0.01),
        "double_well": (sys_dw,   0.5, 0.5,    10, 0.01),
        "pendulum":    (sys_pend, 1.0, 1.0,    30, 0.01),
    }
 
    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 1: Trajectory examples (all 3 systems)
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 1: Trajectory examples\n" + "#"*80)
    traj_methods = ["Autonomous", "CCA (Whitened)", "CCA (Adaptive)"]
    for tag, (sys, a, r_neigh, tau, D_eval) in systems.items():
        run_evaluation_suite(sys, a=a, r_neigh=r_neigh, D_eval=D_eval, dt=dt,
                             num_trajs=100, plot_trajs=3, spread_s=0.75, tau=tau,
                             methods_to_run=traj_methods, use_whitened=use_whitened,
                             file_suffix="trajs")
 
    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 2: Linear system diagnostics
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 2: Linear diagnostics\n" + "#"*80)
    sys, a, r_neigh, tau, D_eval = systems["linear_2d"]
 
    # 2a. Convergence
    conv_csv = "figures/linear_2d_convergence.csv"
    generate_convergence_data(sys, a=a, D_train=D_eval, dt=dt, T_train=500,
                              r_neigh=r_neigh, num_trajs=num_trajs, tau=tau,
                              use_whitened=use_whitened, csv_filepath=conv_csv)
    # FIX: removed legend_fontsize (not in doc 9's signature)
    plot_convergence_from_data(csv_filepath=conv_csv, sys_lambda_true=sys.lambda_u_true,
                               sys_D_true=D_eval, filepath="figures/linear_2d_convergence.pdf",
                               layout="stacked", label_fontsize=13, tick_fontsize=12)
 
    # 2b. Noise sweep — FIX: use plot_scaling_and_energy (not plot_success_and_energy_vs_noise)
    energy_csv = "figures/linear_2d_noise_sweep.csv"
    simulate_and_save_noise_energy_sweep(sys, a=a, dt=dt, r_neigh=r_neigh,
                                         num_trajs=num_trajs, T_sim=2000, tau=tau,
                                         use_whitened=use_whitened, csv_filepath=energy_csv)
    plot_scaling_and_energy(energy_csv, sys_name=sys.name,
                            filepath="figures/linear_2d_noise_sweep.pdf",
                            mode="stacked", include_adaptive=False)
 
    # 2c. Dimensionality & lambda scaling
    run_dimensionality_scaling_experiment(dt=dt, num_trajs=num_trajs, T_sim=5000, use_whitened=use_whitened)
 
    # 2d. Combined scaling + energy — FIX: use plot_combined_scaling_and_energy
    plot_combined_scaling_and_energy(
        dim_csv_filepath="figures/dimensionality_scaling_stats_WHITENED_CCA.csv",
        energy_csv_filepath=energy_csv,
        include_adaptive=False,
        filepath="figures/linear_2d_combined_scaling_energy.pdf")
 
    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 3: Nonlinear systems — noise sweep
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 3: Nonlinear noise sweeps\n" + "#"*80)
    for tag in ["double_well", "pendulum"]:
        sys, a, r_neigh, tau, D_eval = systems[tag]
        csv_path = f"figures/{tag}_noise_sweep.csv"
        simulate_and_save_noise_energy_sweep(sys, a=a, dt=dt, r_neigh=r_neigh,
                                              num_trajs=num_trajs, T_sim=2000, tau=tau,
                                              use_whitened=use_whitened, csv_filepath=csv_path)
        # FIX: use plot_scaling_and_energy (not plot_success_and_energy_vs_noise)
        plot_scaling_and_energy(csv_path, sys_name=sys.name,
                                filepath=f"figures/{tag}_noise_sweep.pdf",
                                mode="stacked", include_adaptive=False)
 
    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 4: Baseline comparisons
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 4: Baseline comparisons\n" + "#"*80)
    for tag in ["linear_2d", "double_well"]:
        sys, a, r_neigh, tau, D_eval = systems[tag]
        csv_path = f"figures/{tag}_baselines.csv"
        run_baseline_comparison(sys, a=a, r_neigh=r_neigh, D_eval=D_eval, dt=dt,
                                num_trajs=500, T_sim=2000, T_train=2500, tau=tau,
                                csv_filepath=csv_path)
        # FIX: removed sys.name positional arg (function doesn't take it)
        plot_baseline_comparison(csv_filepath=csv_path,
                                 filepath=f"figures/{tag}_baselines.pdf")
 
    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 5: Ablations
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 5: Ablations\n" + "#"*80)
 
    # 5a. Trust-region (Neighborhood) Ablation - Reproduces Table I
    for tag in ["double_well", "pendulum"]:
        sys, a, r_neigh, tau, D_eval = systems[tag]
        csv_path = f"figures/{tag}_trust_ablation.csv"
        
        # We use the specific levels from Table I to show the trade-off
        # between data starvation (0.1) and nonlinearity poisoning (inf)
        run_trust_region_ablation(sys, a=a, D_eval=D_eval, dt=dt, num_trajs=500,
                                  T_sim=2000, T_train=2500, tau=tau,
                                  r_neigh_values=[0.1, 0.25, 0.5, 1.0, np.inf], 
                                  csv_filepath=csv_path)
        
        plot_trust_region_ablation(csv_path, sys.name, f"figures/{tag}_trust_ablation.pdf")
 
    # 5b. Estimator Robustness (Lag & Discount Factor)
    sys, a, r_neigh, tau, D_eval = systems["linear_2d"]
    csv_path = "figures/linear_2d_hyperparam_ablation.csv"
    run_hyperparameter_ablation(sys, a=a, r_neigh=r_neigh, D_eval=D_eval, dt=dt,
                                 num_trajs=500, T_sim=2000, T_train=2500,
                                 tau_values=[1, 5, 10, 20, 50],
                                 gamma_values=[0.0, 0.005, 0.01, 0.05, 0.1, .5, 1.0],
                                 csv_filepath=csv_path)
    
    plot_hyperparameter_ablation(csv_path, sys.name, "figures/linear_2d_hyperparam_ablation.pdf",
                                 true_lambda=sys.lambda_u_true)


################################################################################
 FIGURE 1: Trajectory examples
################################################################################

 Evaluating: Linear_2D  (a=1.0, D=0.01, r_neigh=inf) | Mode: WHITENED CCA
  True:           lambda_u=2.000, w_u=[0.99590457 0.09041061]
  CCA (Whitened): lambda_u=2.172, D=0.01042, w_u=[0.75235983 0.65875237]
  Autonomous           Reach A = 20/1000 | Time = 0/1000 | Fail = 980/1000 | E(succ) = 0.000 | E(all) = 0.000
  CCA (Whitened)       Reach A = 974/1000 | Time = 0/1000 | Fail = 26/1000 | E(succ) = 1.846 | E(all) = 1.800


'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
meta NOT subset; don't know how to subset; dropped


Saved Grid multi traj -> figures/Linear_2D_trajs_WHITENED_CCA_multi_traj.png


meta NOT subset; don't know how to subset; dropped


Saved Combined single traj -> figures/Linear_2D_trajs_WHITENED_CCA_single_traj_combined.png

 Evaluating: Curved_Double_Well  (a=0.5, D=0.01, r_neigh=0.5) | Mode: WHITENED CCA
  True:           lambda_u=2.000, w_u=[1. 0.]
  CCA (Whitened): lambda_u=1.417, D=0.00983, w_u=[0.97212301 0.23447145]
  Autonomous           Reach A = 47/1000 | Time = 0/1000 | Fail = 953/1000 | E(succ) = 0.000 | E(all) = 0.000
  CCA (Whitened)       Reach A = 992/1000 | Time = 0/1000 | Fail = 8/1000 | E(succ) = 0.385 | E(all) = 0.382


...

Saved Combined single traj -> figures/Pendulum_trajs_WHITENED_CCA_single_traj_combined.png

################################################################################
 FIGURE 3: Nonlinear noise sweeps
################################################################################


'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
meta NOT subset; don't know how to subset; dropped


  Saved -> figures/double_well_noise_sweep.pdf


'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
meta NOT subset; don't know how to subset; dropped


  Saved -> figures/pendulum_noise_sweep.pdf
